# Phase 4B — IEEE-CIS Dataset
**Goal:** Train XGBoost on IEEE-CIS dataset and compare performance with ULB model.

In [ ]:
import pandas as pd
import numpy as np
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, f1_score, roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

print("Loading IEEE-CIS dataset...")
txn = pd.read_csv('../data/ieee/train_transaction.csv')
idn = pd.read_csv('../data/ieee/train_identity.csv')
df_ieee = txn.merge(idn, on='TransactionID', how='left')
print(f"Merged shape: {df_ieee.shape}")

In [ ]:
# Keep numeric columns only
df_numeric = df_ieee.select_dtypes(include=[np.number]).copy()
y_ieee = df_numeric['isFraud'].copy()
X_ieee = df_numeric.drop(columns=['isFraud','TransactionID'], errors='ignore')

# Drop columns with > 50% nulls
null_pct     = X_ieee.isnull().mean()
cols_to_drop = null_pct[null_pct > 0.5].index.tolist()
X_ieee       = X_ieee.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} high-null columns")

# Fill remaining nulls
X_ieee = X_ieee.fillna(X_ieee.median()).fillna(0)
print(f"Remaining NaN: {X_ieee.isnull().sum().sum()}")
print(f"Features: {X_ieee.shape[1]} | Fraud ratio: {y_ieee.mean()*100:.3f}%")

In [ ]:
# Scale and split
scaler_ieee = StandardScaler()
if 'TransactionAmt' in X_ieee.columns:
    X_ieee['TransactionAmt'] = scaler_ieee.fit_transform(X_ieee[['TransactionAmt']])

X_tr, X_te, y_tr, y_te = train_test_split(
    X_ieee, y_ieee, test_size=0.2, random_state=42, stratify=y_ieee)

# SMOTE
print("Applying SMOTE...")
smote = SMOTE(random_state=42, k_neighbors=3)
X_tr_bal, y_tr_bal = smote.fit_resample(X_tr, y_tr)
print(f"After SMOTE — Normal: {(y_tr_bal==0).sum()} | Fraud: {(y_tr_bal==1).sum()}")

In [ ]:
# Train XGBoost on IEEE
fraud_count  = int(y_tr_bal.sum())
normal_count = int(len(y_tr_bal) - fraud_count)

xgb_ieee = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=normal_count/fraud_count,
    eval_metric='logloss', random_state=42, n_jobs=-1)

print("Training XGBoost on IEEE (3-5 mins)...")
xgb_ieee.fit(X_tr_bal, y_tr_bal)
print("Done!")

In [ ]:
# Evaluate
y_pred      = xgb_ieee.predict(X_te)
y_pred_prob = xgb_ieee.predict_proba(X_te)[:,1]

print("=== IEEE Dataset Results ===")
print(f"Recall  : {recall_score(y_te, y_pred):.4f}")
print(f"F1      : {f1_score(y_te, y_pred):.4f}")
print(f"ROC-AUC : {roc_auc_score(y_te, y_pred_prob):.4f}")
print("\nDetailed Report:")
print(classification_report(y_te, y_pred, target_names=['Normal','Fraud']))

In [ ]:
# Compare ULB vs IEEE
from sklearn.metrics import recall_score, f1_score, roc_auc_score
X_test_cc = joblib.load('../data/X_test.pkl')
y_test_cc  = joblib.load('../data/y_test.pkl')
xgb_cc     = joblib.load('../models/xgboost_model.pkl')

y_pred_cc       = xgb_cc.predict(X_test_cc)
y_pred_prob_cc  = xgb_cc.predict_proba(X_test_cc)[:,1]

comparison = pd.DataFrame({
    'Dataset' : ['ULB Credit Card','IEEE-CIS'],
    'Recall'  : [round(recall_score(y_test_cc,y_pred_cc),4), round(recall_score(y_te,y_pred),4)],
    'F1'      : [round(f1_score(y_test_cc,y_pred_cc),4),     round(f1_score(y_te,y_pred),4)],
    'ROC-AUC' : [round(roc_auc_score(y_test_cc,y_pred_prob_cc),4), round(roc_auc_score(y_te,y_pred_prob),4)],
})
print("=== MULTI-DATASET COMPARISON ===")
print(comparison.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(3); width = 0.3
ax.bar(x-width/2, comparison.loc[0,['Recall','F1','ROC-AUC']], width, label='ULB Credit Card', color='steelblue', alpha=0.85)
ax.bar(x+width/2, comparison.loc[1,['Recall','F1','ROC-AUC']], width, label='IEEE-CIS',        color='crimson',   alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(['Recall','F1','ROC-AUC'])
ax.set_ylim(0,1.1); ax.set_title('XGBoost — ULB vs IEEE-CIS')
ax.legend(); plt.tight_layout()
plt.savefig('../reports/phase4b_multi_dataset_comparison.png', dpi=150)
plt.show()

In [ ]:
# Save IEEE model
joblib.dump(xgb_ieee, '../models/xgboost_ieee_model.pkl')
print("IEEE model saved!")
print("Phase 4B complete!")